# INV-M365-CPS-A1-T2 — Status Code Semantics v1

**Lemma:** `L100_m365_cps_status_code_semantics_v1`
**Plan:** `plan:m365-cps-trkA-p1-status-code-semantics:T2`
**Target:** `src/m365_runtime/graph/actions.py` `_denial_to_status` line 129

## Mission

Make `_denial_to_status` semantically honest. The pre-fix implementation collapsed `unknown_action` into `mutation_fence`, which lied about denials. This notebook proves the bundled predicate `StatusDenialHonest` over the post-fix function:

```
StatusDenialHonest =
    L_PRESERVE_PERMISSION_MISSING
  ∧ L_PRESERVE_AUTH_REQUIRED
  ∧ L_FIX_UNKNOWN_ACTION
  ∧ L_PRESERVE_MUTATION_FENCE
  ∧ L_PRESERVE_DEFAULT
```

## Cell 1 — Seed lock and context

Deterministic-static. No randomness, no live network. Seed locked at 0 by convention.

In [ ]:
import random
import sys
from pathlib import Path

random.seed(0)

# Make src/ importable
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'src' / 'm365_runtime').is_dir() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from m365_runtime.graph.actions import _denial_to_status
print('imported _denial_to_status from', _denial_to_status.__module__)

## Cell 2 — L_PRESERVE_PERMISSION_MISSING

Input `permission_missing` must return `permission_missing` (unchanged).

In [ ]:
L_PRESERVE_PERMISSION_MISSING = _denial_to_status('permission_missing') == 'permission_missing'
assert L_PRESERVE_PERMISSION_MISSING, 'permission_missing branch regressed'
print('L_PRESERVE_PERMISSION_MISSING:', L_PRESERVE_PERMISSION_MISSING)

## Cell 3 — L_PRESERVE_AUTH_REQUIRED

Input `auth_mode_mismatch` must return `auth_required` (unchanged).

In [ ]:
L_PRESERVE_AUTH_REQUIRED = _denial_to_status('auth_mode_mismatch') == 'auth_required'
assert L_PRESERVE_AUTH_REQUIRED, 'auth_mode_mismatch branch regressed'
print('L_PRESERVE_AUTH_REQUIRED:', L_PRESERVE_AUTH_REQUIRED)

## Cell 4 — L_FIX_UNKNOWN_ACTION (the change)

Input `unknown_action` must return `unknown_action`. This is the bug fix — pre-fix returned `mutation_fence`.

In [ ]:
L_FIX_UNKNOWN_ACTION = _denial_to_status('unknown_action') == 'unknown_action'
assert L_FIX_UNKNOWN_ACTION, 'unknown_action branch did not return unknown_action; check actions.py:135'
# Also assert the prior bug is gone
assert _denial_to_status('unknown_action') != 'mutation_fence', 'unknown_action still collapsed to mutation_fence'
print('L_FIX_UNKNOWN_ACTION:', L_FIX_UNKNOWN_ACTION)

## Cell 5 — L_PRESERVE_MUTATION_FENCE

Input `mutation_fence` must return `mutation_fence` (unchanged). Genuine writes still get fenced.

In [ ]:
L_PRESERVE_MUTATION_FENCE = _denial_to_status('mutation_fence') == 'mutation_fence'
assert L_PRESERVE_MUTATION_FENCE, 'mutation_fence branch regressed'
print('L_PRESERVE_MUTATION_FENCE:', L_PRESERVE_MUTATION_FENCE)

## Cell 6 — L_PRESERVE_DEFAULT

Any other input must return `policy_denied` (default branch unchanged).

In [ ]:
_default_inputs = ['', 'foo', 'bar', 'anything_else']
_default_results = [_denial_to_status(x) for x in _default_inputs]
L_PRESERVE_DEFAULT = all(r == 'policy_denied' for r in _default_results)
assert L_PRESERVE_DEFAULT, f'default branch regressed: {dict(zip(_default_inputs, _default_results))}'
print('L_PRESERVE_DEFAULT:', L_PRESERVE_DEFAULT)

## Cell 7 — Conjunction StatusDenialHonest

All five lemmas must hold.

In [ ]:
StatusDenialHonest = (
    L_PRESERVE_PERMISSION_MISSING
    and L_PRESERVE_AUTH_REQUIRED
    and L_FIX_UNKNOWN_ACTION
    and L_PRESERVE_MUTATION_FENCE
    and L_PRESERVE_DEFAULT
)
assert StatusDenialHonest, 'StatusDenialHonest conjunction failed; check individual lemmas above'
print('StatusDenialHonest:', StatusDenialHonest)

## Cell 8 — Truth-table proof (32 rows)

Enumerate all 2^5 = 32 truth-table rows. The conjunction is true on exactly 1 row (all-true).

In [ ]:
import itertools
rows = list(itertools.product([False, True], repeat=5))
true_rows = [r for r in rows if all(r)]
false_rows = [r for r in rows if not all(r)]
assert len(rows) == 32
assert len(true_rows) == 1
assert len(false_rows) == 31
print(f'truth_table_rows={len(rows)} true_rows={len(true_rows)} false_rows={len(false_rows)}')

## Cell 9 — Emit scorecard

In [ ]:
import json
scorecard = {
    'status': 'green',
    'lemma_id': 'L100_m365_cps_status_code_semantics_v1',
    'plan_reference': 'plan:m365-cps-trkA-p1-status-code-semantics:T2',
    'notebook_reference': 'notebooks/m365/INV-M365-CPS-A1-T2-status-code-semantics-v1.ipynb',
    'generated_verification_reference': 'configs/generated/m365_cps_status_code_semantics_v1_verification.json',
    'notebook_hash_mode': 'static-governed',
    'seed_lock': {'enabled': True, 'seed': 0, 'mode': 'deterministic-static'},
    'invariants': [
        {'id': 'L_PRESERVE_PERMISSION_MISSING', 'status': 'pass' if L_PRESERVE_PERMISSION_MISSING else 'fail'},
        {'id': 'L_PRESERVE_AUTH_REQUIRED', 'status': 'pass' if L_PRESERVE_AUTH_REQUIRED else 'fail'},
        {'id': 'L_FIX_UNKNOWN_ACTION', 'status': 'pass' if L_FIX_UNKNOWN_ACTION else 'fail'},
        {'id': 'L_PRESERVE_MUTATION_FENCE', 'status': 'pass' if L_PRESERVE_MUTATION_FENCE else 'fail'},
        {'id': 'L_PRESERVE_DEFAULT', 'status': 'pass' if L_PRESERVE_DEFAULT else 'fail'},
        {'id': 'StatusDenialHonest', 'status': 'pass' if StatusDenialHonest else 'fail'},
    ],
    'negative_case_count': len(false_rows[:4]),
    'truth_table_rows': len(rows),
    'truth_table_true_rows': len(true_rows),
    'truth_table_false_rows': len(false_rows),
    'determinism_certification': 'green',
    'ready_for_tasks': ['T3', 'T4', 'T5', 'T6', 'T7', 'T8'],
}
print(json.dumps(scorecard, indent=2))